# Aprendizaje *out-of-core*

## Problemas de escalabilidad

Las clases `sklearn.feature_extraction.text.CountVectorizer` y `sklearn.feature_extraction.text.TfidfVectorizer` tienen una serie de problemas de escalabilidad que provienen de la forma en que se utiliza, a nivel interno, el atributo `vocabulary_` (que es un diccionario Python) para convertir los nombres de las características (cadenas) a índices enteros de características.

Los principales problemas de escalabilidad son:

- **Uso de memoria del vectorizador de texto**: todas las representaciones textuales de características se cargan en memoria.
- **Problemas de paralelización para extracción de características**: el atributo `vocabulary_` es compartido, lo que conlleva que sea difícil la sincronización y por tanto que se produzca una sobrecarga.
- **Imposibilidad de realizar aprendizaje *online*, *out-of-core* o *streaming***: el atributo `vocabulary_` tiene que obtenerse a partir de los datos y su tamaño no se puede conocer hasta que no realizamos una pasada completa por toda la base de datos de entrenamiento.

Para entender mejor estos problemas, analicemos como trabaja el atributo `vocabulary_`. En la fase de `fit` se identifican los tokens del corpus de forma unívoca, mediante un índice entero, y esta correspondencia se guarda en el vocabulario:

In [1]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(min_df=1)

vectorizer.fit([
    "The cat sat on the mat.",
])
vectorizer.vocabulary_

{'the': 4, 'cat': 0, 'sat': 3, 'on': 2, 'mat': 1}

El vocabulario se utiliza en la fase `transform` para construir la matriz de ocurrencias:

In [3]:
X = vectorizer.transform([
    "The cat sat on the mat.",
    "This cat is a nice cat.",
]).toarray()

print(len(vectorizer.vocabulary_))
print(vectorizer.get_feature_names_out())
print(X)

5
['cat' 'mat' 'on' 'sat' 'the']
[[1 1 1 1 2]
 [2 0 0 0 0]]


Vamos a realizar un nuevo `fit` con un corpus algo más grande:

In [4]:
vectorizer = CountVectorizer(min_df=1)

vectorizer.fit([
    "The cat sat on the mat.",
    "The quick brown fox jumps over the lazy dog.",
])
vectorizer.vocabulary_

{'the': 11,
 'cat': 1,
 'sat': 10,
 'on': 7,
 'mat': 6,
 'quick': 9,
 'brown': 0,
 'fox': 3,
 'jumps': 4,
 'over': 8,
 'lazy': 5,
 'dog': 2}

El atributo `vocabulary_` crece (en escala logarítmica) con respecto al tamaño del conjunto de entrenamiento. Observa que no podemos construir los vocabularios en paralelo para cada documento de texto ya que hay algunas palabras que son comunes y necesitaríamos alguna estructura compartida o barrera de sincronización (aumentando la complejidad de implementar el entrenamiento, sobre todo si queremos distribuirlo en un cluster).

Con este nuevo vocabulario, la dimensionalidad del espacio de salida es mayor:

In [6]:
X = vectorizer.transform([
    "The cat sat on the mat.",
    "This cat is a nice cat.",
]).toarray()

print(len(vectorizer.vocabulary_))
print(vectorizer.get_feature_names_out())
print(X)

12
['brown' 'cat' 'dog' 'fox' 'jumps' 'lazy' 'mat' 'on' 'over' 'quick' 'sat'
 'the']
[[0 1 0 0 0 0 1 1 0 0 1 2]
 [0 2 0 0 0 0 0 0 0 0 0 0]]


## El dataset de películas IMDb

Para mostrar los problemas de escalabilidad con los vocabularios basados en vectorizadores, vamos a cargar un dataset realista que proviene de una tarea típica de clasificación de textos: análisis de sentimientos en texto. El objetivo es discernir entre revisiones positivas y negativas a partir de la base de datos de [Internet Movie Database](http://www.imdb.com) (IMDb).

En las siguientes secciones, vamos a usar el siguiente dataset [subset](http://ai.stanford.edu/~amaas/data/sentiment/) de revisiones de películas de IMDb, que has sido recolectado por Maas et al. 

- A. L. Maas, R. E. Daly, P. T. Pham, D. Huang, A. Y. Ng, and C. Potts. Learning Word Vectors for Sentiment Analysis. In the proceedings of the 49th Annual Meeting of the Association for Computational Linguistics: Human Language Technologies, pages 142–150, Portland, Oregon, USA, June 2011. Association for Computational Linguistics. 

Este dataset contiene 50,000 revisiones de películas, divididas en 25,000 ejemplos de entrenamiento y 25,000 ejemplos de test. Las revisiones se etiquetan como negativas (`neg`) o positivas (`pos`). De hecho, las negativas recibieron $\le 4$ estrellas en IMDb; las positivas recibieron $\ge 7$ estrellas. Las revisiones neutrales no se incluyeron en el dataset.

Asumiendo que ya habéis ejecutado el *script* `fetch_data.py`, deberías tener disponibles los siguientes ficheros:

In [7]:
import os

train_path = os.path.join('datasets', 'IMDb', 'aclImdb', 'train')
test_path = os.path.join('datasets', 'IMDb', 'aclImdb', 'test')

Ahora, vamos a cargarlos en nuestra sesión activa usando la función `load_files` de scikit-learn:

In [ ]:
from sklearn.datasets import load_files

# Cargar el conjunto de entrenamiento desde el directorio de IMDb
# Se limitan las categorias a 'pos' y 'neg'
train = load_files(container_path=(train_path),
                   categories=['pos', 'neg'])

# Cargar el conjunto de test desde el directorio de IMDb
# Se limitan las categorias a 'pos' y 'neg'
test = load_files(container_path=(test_path),
                  categories=['pos', 'neg'])

<div class="alert alert-warning">
    <b>NOTA</b>:
     <ul>
      <li>
      Ya que el dataset de películas contiene 50,000 ficheros individuales de texto, ejecutar el código anterior puede llevar bastante tiempo.
      </li>
    </ul>
</div>

La función `load_files`  ha cargado los datasets en objetos `sklearn.datasets.base.Bunch`, que son diccionarios de Python:

In [9]:
train.keys()

dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])

En particular, solo estamos interesados en los arrays `data` y `target`.

In [10]:
import numpy as np

for label, data in zip(('ENTRENAMIENTO', 'TEST'), (train, test)):
    print('\n\n%s' % label)
    print('Número de documentos:', len(data['data']))
    print('\n1er documento:\n', data['data'][0])
    print('\n1era etiqueta:', data['target'][0])
    print('\nNombre de las clases:', data['target_names'])
    print('Conteo de las clases:', 
          np.unique(data['target']), ' -> ',
          np.bincount(data['target']))



ENTRENAMIENTO
Número de documentos: 25000

1er documento:
 b"Zero Day leads you to think, even re-think why two boys/young men would do what they did - commit mutual suicide via slaughtering their classmates. It captures what must be beyond a bizarre mode of being for two humans who have decided to withdraw from common civility in order to define their own/mutual world via coupled destruction.<br /><br />It is not a perfect movie but given what money/time the filmmaker and actors had - it is a remarkable product. In terms of explaining the motives and actions of the two young suicide/murderers it is better than 'Elephant' - in terms of being a film that gets under our 'rationalistic' skin it is a far, far better film than almost anything you are likely to see. <br /><br />Flawed but honest with a terrible honesty."

1era etiqueta: 1

Nombre de las clases: ['neg', 'pos']
Conteo de las clases: [0 1]  ->  [12500 12500]


TEST
Número de documentos: 25000

1er documento:
 b"Don't hate Hea

Como puedes comprobar, el array `'target'` consiste en valores `0` y `1`, donde el `0` es una revisión negativa y el `1` representa una positiva.

## El truco del *hashing*

Recuerda la representación *bag-of-words* que se obtenía usando un vectorizador basado en vocabulario:

<img src="figures/bag_of_words.svg" width="100%">

Para solventar las limitaciones de los vectorizadores basados en vocabularios, se puede utilizar el truco del *hashing*. En lugar de construir y almacenar una conversión explícita de los nombres de las características a los índices de las mismas dentro de un diccionario Python, podemos aplicar una función de *hash* y el operador módulo:

<img src="figures/hashing_vectorizer.svg" width="100%">

Podemos acceder a más información y a las referencias a los artículos originales en el siguiente [sitio web](http://www.hunch.net/~jl/projects/hash_reps/index.html), y una descripción más sencilla en este otro [sitio](http://blog.someben.com/2013/01/hashing-lang/).

In [11]:
from sklearn.utils.murmurhash import murmurhash3_bytes_u32

# Codificado para compatibilidad con Python 3
for word in "the cat sat on the mat".encode("utf-8").split():
    print("{0} => {1}".format(
        word, murmurhash3_bytes_u32(word, 0) % 2 ** 20))

b'the' => 761698
b'cat' => 300839
b'sat' => 122804
b'on' => 735689
b'the' => 761698
b'mat' => 122997


La conversión no tiene estado y la dimensionalidad del espacio de salida se fija a priori (aquí usamos módulo `2 ** 20`, que significa aproximadamente que tenemos un millón de dimensiones, $2^{20}$). Esto hace posible evitar las limitaciones del vectorizador de vocabulario, tanto a nivel de paralelización como de poder aplicar aprendizaje *online*.

La clase `HashingVectorizer` es una alternativa a `CountVectorizer` (o a `TfidfVectorizer` si consideramos `use_idf=False`) que aplica internamente la función de *hash* llamada ``murmurhash``:

In [12]:
from sklearn.feature_extraction.text import HashingVectorizer

h_vectorizer = HashingVectorizer(encoding='latin-1')
h_vectorizer

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'latin-1'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any character.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'


Comparte la misma estructura de preprocesamiento, generación de tokens y análisis:

In [13]:
analyzer = h_vectorizer.build_analyzer()
analyzer('Esta es una frase de prueba.')

['esta', 'es', 'una', 'frase', 'de', 'prueba']

Podemos vectorizar nuestros datasets en matriz dispersa de scipy de la misma forma que hubiéramos hecho con `CountVectorizer` o `TfidfVectorizer`, excepto que podemos llamar directamente al método `transform`. No hay necesidad de llamar a `fit` porque el `HashingVectorizer` no se entrena, las transformaciones están prefijadas.

In [14]:
docs_train, y_train = train['data'], train['target']
docs_valid, y_valid = test['data'][:12500], test['target'][:12500]
docs_test, y_test = test['data'][12500:], test['target'][12500:]

La dimensión de salida se fija de antemano a `n_features=2 ** 20` (valor por defecto) para minimizar la probabilidad de colisión en la mayoría de problemas de clasificación (1M de pesos en el atributo `coef_`):

In [15]:
h_vectorizer.transform(docs_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3446628 stored elements and shape (25000, 1048576)>

Ahora vamos a comparar la eficiencia computacional de `HashingVectorizer` con respecto a `CountVectorizer`:

In [16]:
h_vec = HashingVectorizer(encoding='latin-1')
%timeit -n 1 -r 3 h_vec.fit(docs_train, y_train)

93.6 μs ± 29 μs per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [17]:
count_vec =  CountVectorizer(encoding='latin-1')
%timeit -n 1 -r 3 count_vec.fit(docs_train, y_train)

2.56 s ± 7.32 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


Como puedes observar, ``HashingVectorizer`` es mucho más rápido que ``Countvectorizer``.

Por último, vamos a entrenar un clasificador ``LogisticRegression`` en los datos de entrenamiento de IMDb:

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

h_pipeline = Pipeline([
    ('vec', HashingVectorizer(encoding='latin-1')),
    ('clf', LogisticRegression(random_state=1)),
])

h_pipeline.fit(docs_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vec', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'latin-1'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any character.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [19]:
print('Accuracy de entrenamiento', h_pipeline.score(docs_train, y_train))
print('Accuracy de validación', h_pipeline.score(docs_valid, y_valid))

Accuracy de entrenamiento 0.87844
Accuracy de validación 0.8624


In [20]:
import gc

del count_vec
del h_pipeline

gc.collect()

895

# Aprendizaje *Out-of-Core*

El aprendizaje *Out-of-Core* consiste en entrenar un modelo de aprendizaje automático usando un dataset que no cabe en memoria RAM. Requiere las siguientes condiciones:
    
- Una capa de **extracción de características** con una **dimensionalidad de salida fija**.
- Saber la lista de clases de antemano (en este caso, sabemos que hay *tweets* positivos y negativos).
- Un algoritmo de aprendizaje automático que soporte **aprendizaje incremental** (método `partial_fit` en scikit-learn).

En la siguientes secciones, vamos a configurar una función simple de entrenamiento iterativo de un `SGDClassifier`.

Pero primero cargamos los nombres de los ficheros en una lista de Python:

In [21]:
train_path = os.path.join('datasets', 'IMDb', 'aclImdb', 'train')
train_pos = os.path.join(train_path, 'pos')
train_neg = os.path.join(train_path, 'neg')

fnames = [os.path.join(train_pos, f) for f in os.listdir(train_pos)] +\
         [os.path.join(train_neg, f) for f in os.listdir(train_neg)]

fnames[:3]

['datasets/IMDb/aclImdb/train/pos/7506_7.txt',
 'datasets/IMDb/aclImdb/train/pos/2232_7.txt',
 'datasets/IMDb/aclImdb/train/pos/363_10.txt']

Ahora vamos a crear el array de etiquetas:

In [22]:
y_train = np.zeros((len(fnames), ), dtype=int)
y_train[:12500] = 1
np.bincount(y_train)

array([12500, 12500])

Ahora vamos a implementar la función `batch_train function`:

In [23]:
from sklearn.base import clone

def batch_train(clf, fnames, labels, iterations=25, batchsize=1000, random_seed=1):
    vec = HashingVectorizer(encoding='latin-1')
    idx = np.arange(labels.shape[0])
    c_clf = clone(clf)
    rng = np.random.RandomState(seed=random_seed)
    
    for i in range(iterations):
        rnd_idx = rng.choice(idx, size=batchsize)
        documents = []
        for i in rnd_idx:
            with open(fnames[i], 'r') as f:
                documents.append(f.read())
        X_batch = vec.transform(documents)
        batch_labels = labels[rnd_idx]
        c_clf.partial_fit(X=X_batch, 
                          y=batch_labels, 
                          classes=[0, 1])
      
    return c_clf

Ahora vamos a utilizar la clase un `SGDClassifier` con un coste logístico en lugar de `LogisticRegression`. SGD proviene de *stochastic gradient descent*, un algoritmo de optimización que optimiza los pesos de forma iterativa ejemplo a ejemplo, lo que nos permite pasarle los datos en grupos.

Como empleamos el `SGDClassifier` con la configuración por defecto, entrenará el clasificador en 25\*1000=25000 documentos (lo que puede llevar algo de tiempo).

In [25]:
from sklearn.linear_model import SGDClassifier

sgd = SGDClassifier(loss='log_loss', random_state=1)

sgd = batch_train(clf=sgd,
                  fnames=fnames,
                  labels=y_train)

Al terminar, evaluemos el rendimiento:

In [26]:
vec = HashingVectorizer(encoding='latin-1')
sgd.score(vec.transform(docs_test), y_test)

0.8252

### Limitaciones de los vectorizadores basados en *hash*

Utilizar este tipo de vectorizadores nos permiten una mejor escalabilidad y trabajar *on-line*, pero también tienen algunos inconvenientes:
    
- Las colisiones podrían introducir ruido en los datos y degradar la calidad de las predicciones.
- La clase `HashingVectorizer` no nos permite emplear "*Inverse Document Frequency*" (no existe la opción `use_idf=True`).
- No hay una forma simple de realizar una conversión inversa y encontrar los nombres de las características a partir del índice.

Las colisiones pueden minimizarse incrementando el parámetro `n_features`.

El peso IDF podría reintroducirse si añadimos un objeto de la clase `TfidfTransformer` a la salida del vectorizador. Sin embargo, obtener el estadístico `idf_` utilizado para sopesar las características implicaría al menos una pasada adicional por los datos de entrenamiento antes de empezar a entrenar el clasificador: ya no podríamos afrontar un escenario *on-line*.

La falta de una conversión inversa (método `get_feature_names()` de `TfidfVectorizer`) es mucho más difícil de evitar. Tendríamos que extender la clase `HashingVectorizer` para añadir un modo "traza" que nos permitiera guardar la conversión de las características más importantes.

<div class="alert alert-success">
    <b>EJERCICIO</b>:
     <ul>
      <li>
      En la implementación propuesta de la función ``batch_train``, se tomaron aleatoriamente *k* ejemplos de entrenamiento como *batch* en cada iteración, lo que puede considerarse un muestreo aleatorio con reemplazamiento. Modifica la función `batch_train` de forma que itere sobre todos los documentos ***sin*** reemplazamiento, es decir, que se usa cada documento ***una sola vez*** por iteración?
      </li>
    </ul>
</div>